In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

# ==== Load base JAAD data ====
print("Loading CSV data...")
df = pd.read_csv('jaad_extracted_data.csv')

# ==== Bounding box geometry ====
W, H = 1920.0, 1080.0
df = df.sort_values(['video_id', 'pedestrian_id', 'frame_id']).reset_index(drop=True)

df['bbox_height']   = (df['bbox_y2'] - df['bbox_y1']) / H
df['bbox_width']    = (df['bbox_x2'] - df['bbox_x1']) / W
df['bbox_center_x'] = ((df['bbox_x1'] + df['bbox_x2']) / 2.0) / W
df['bbox_center_y'] = ((df['bbox_y1'] + df['bbox_y2']) / 2.0) / H
df['bbox_area']     = df['bbox_width'] * df['bbox_height']

# ==== Create unique track identifier ====
df['track_id'] = df['video_id'].astype(str) + "_" + df['pedestrian_id'].astype(str)

# ==== Motion features (per pedestrian track deltas) ====
gkeys = ['video_id', 'pedestrian_id']
df['velocity_x']     = df.groupby(gkeys)['bbox_center_x'].diff().fillna(0.0)
df['velocity_y']     = df.groupby(gkeys)['bbox_center_y'].diff().fillna(0.0)
df['speed']          = np.sqrt(df['velocity_x']**2 + df['velocity_y']**2)
df['acceleration_x'] = df.groupby(gkeys)['velocity_x'].diff().fillna(0.0)
df['acceleration_y'] = df.groupby(gkeys)['velocity_y'].diff().fillna(0.0)


df['rolling_velocity_x'] = df.groupby(gkeys)['velocity_x'].transform(lambda x: x.rolling(10, min_periods=1).mean())
df['rolling_velocity_y'] = df.groupby(gkeys)['velocity_y'].transform(lambda x: x.rolling(10, min_periods=1).mean())

# Keep the original labels for safety evaluation
df['crossing_actual'] = df['cross']

# Create the shifted target for training
anticipation_frames = 60
df['crossing_shifted'] = df.groupby(gkeys)['cross'].shift(-anticipation_frames)

# ==== Load and unpack pose features ====
print("Loading and unpacking pose data...")
with open('pedestrian_poses.pkl', 'rb') as f:
    poses = pickle.load(f)

pose_rows = []
for track_id, frames in poses.items():
    for fd in frames:
        kpts_xy   = fd.get('keypoints_xy', [])
        kpts_conf = fd.get('keypoints_conf', [])

        visible_kpts = sum(1 for c in kpts_conf if c > 0.3) if kpts_conf else 0
        mean_conf    = float(np.mean(kpts_conf))  if kpts_conf else 0.0
        xs = [x for x, y in kpts_xy if x > 0]    if kpts_xy else []
        ys = [y for x, y in kpts_xy if y > 0]    if kpts_xy else []
        x_span = float(max(xs) - min(xs)) if len(xs) > 1 else 0.0
        y_span = float(max(ys) - min(ys)) if len(ys) > 1 else 0.0

        pose_rows.append({
            'video_id':          fd['video_id'],
            'pedestrian_id':     fd['pedestrian_id'],
            'frame_id':          fd['frame_id'],
            'pose_visible_kpts': visible_kpts,
            'pose_mean_conf':    mean_conf,
            'pose_x_span':       x_span,
            'pose_y_span':       y_span,
        })

poses_df = pd.DataFrame(pose_rows)

# ==== Merge pose into main dataframe ====
print("Merging datasets...")
df = pd.merge(df, poses_df, on=['video_id', 'pedestrian_id', 'frame_id'], how='left')
for col in ['pose_visible_kpts', 'pose_mean_conf', 'pose_x_span', 'pose_y_span']:
    df[col] = df[col].fillna(0.0)

# Drop NaN targets caused by the shift
df = df.dropna(subset=['crossing_shifted']).copy()

# ==== Feature columns ====
cat_features = [
    'occlusion', 'reaction', 'hand_gesture', 'look', 'action', 'nod',
    'age', 'designated', 'gender', 'intersection',
    'motion_direction', 'signalized', 'traffic_direction'
]
num_features = [
    'bbox_center_x', 'bbox_center_y', 'bbox_width', 'bbox_height', 'bbox_area',
    'velocity_x', 'velocity_y', 'speed', 'acceleration_x', 'acceleration_y',
    'rolling_velocity_x', 'rolling_velocity_y', 
    'group_size', 'num_lanes',
    'pose_visible_kpts', 'pose_mean_conf', 'pose_x_span', 'pose_y_span'
]

# Ensure we only use valid cross labels
df = df[df['crossing_shifted'].isin([0, 1])].copy()

X = df[cat_features + num_features]
# Train entirely on the shifted anticipation target
y = df['crossing_shifted']
groups = df['track_id']

# ==== GroupShuffleSplit to prevent data leakage ====
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# ==== Preprocessing pipeline ====
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
    ('num', StandardScaler(), num_features),
])

# ==== Calibrated Random Forest with Custom Weights ====
print("Training Advanced Random Forest Classifier...")

# Penalize a missed crossing 8x more than a false alarm
custom_weights = {0: 1.0, 1: 8.0} 

base_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=5,
    class_weight=custom_weights,   
    n_jobs=-1,
    random_state=42,
)

# Fix probability shrinkage
calibrated_rf = CalibratedClassifierCV(base_rf, method='isotonic', cv=3)

model = Pipeline([
    ('pre', preprocessor),
    ('clf', calibrated_rf)
])

model.fit(X_train, y_train)

# ==== Generate predictions and save ====
print("Generating advanced predictions...")
df['advanced_prediction'] = model.predict_proba(X[cat_features + num_features])[:, 1]

output_cols = [
    'video_id', 'pedestrian_id', 'frame_id',
    'bbox_y1', 'bbox_y2', 'bbox_height',
    'crossing_actual', 'crossing_shifted', 'advanced_prediction'
]
df[output_cols].to_csv('advanced_predictions.csv', index=False)
print("Model trained! Saved advanced_predictions.csv")

Loading CSV data...
Loading and unpacking pose data...
Merging datasets...
Training Advanced Random Forest Classifier...
Generating advanced predictions...
Model trained! Saved advanced_predictions.csv
